# FashionMNIST Image Classifier

Implementation of the Chapter 10 image classifier workflow. It uses a fixed 55,000/5,000 split of the official training set and keeps the official test set separate.

## 1. Download data, split, and create DataLoaders

Images are converted to tensors with `ToTensor`. The validation split is reproducible.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_data(batch_size=32, data_dir="data"):
    transform = transforms.ToTensor()
    train_all = datasets.FashionMNIST(data_dir, train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(data_dir, train=False, download=True, transform=transform)
    train_set, validation_set = random_split(train_all, [55000, 5000], generator=torch.Generator().manual_seed(42))
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(validation_set, batch_size=batch_size, shuffle=False),
            DataLoader(test_set, batch_size=batch_size, shuffle=False))

train_loader, validation_loader, test_loader = load_data()
device = get_device()
print(f"Device: {device}; train={len(train_loader.dataset)}, validation={len(validation_loader.dataset)}, test={len(test_loader.dataset)}")


## 2. Define the neural network

The model flattens each 28×28 image, applies two ReLU hidden layers, and returns ten raw class logits.

In [ ]:
from torch import nn

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(28 * 28, 300), nn.ReLU(),
            nn.Linear(300, 100), nn.ReLU(), nn.Linear(100, 10))
    def forward(self, images):
        return self.network(images)

model = ImageClassifier().to(device)
model


## 3. Train with SGD

Cross-entropy consumes raw logits. Each epoch records average training loss, training accuracy, and validation accuracy. The trained weights and history are saved for the following sections.

In [ ]:
import json

def multiclass_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
history = {"training_loss": [], "training_accuracy": [], "validation_accuracy": []}
epochs = 20
for epoch in range(epochs):
    model.train()
    loss_sum = correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        n = labels.size(0)
        loss_sum += loss.item() * n
        correct += (logits.argmax(1) == labels).sum().item()
        total += n
    history["training_loss"].append(loss_sum / total)
    history["training_accuracy"].append(correct / total)
    history["validation_accuracy"].append(multiclass_accuracy(model, validation_loader))
    print(f"Epoch {epoch + 1:02d}/{epochs}: loss={history["training_loss"][-1]:.4f}, train_acc={history["training_accuracy"][-1]:.4f}, val_acc={history["validation_accuracy"][-1]:.4f}")
torch.save(model.state_dict(), "model.pth")
with open("history.json", "w", encoding="utf-8") as f: json.dump(history, f, indent=2)


## 4. Plot accuracy

Plot training and validation accuracy against epoch and save the figure as `accuracy.png`.

In [ ]:
import matplotlib.pyplot as plt
epoch_numbers = range(1, len(history["training_accuracy"]) + 1)
plt.plot(epoch_numbers, history["training_accuracy"], label="Training accuracy")
plt.plot(epoch_numbers, history["validation_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy.png", dpi=150)
plt.show()


## 5. Evaluate on the held-out test set

The final accuracy is computed only on the official 10,000-image test split.

In [ ]:
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.size(0)
test_accuracy = correct / total
print(f"Test accuracy: {test_accuracy:.4%}")
